# Séance 9 — Agents IA avec LangChain
### Cours "Visualisation de Données"

---

> **🎯 Objectifs de la séance**
> - Construire des agents capables d'analyser `job_postings.csv` de façon autonome
> - Définir des tools réutilisables avec le décorateur `@tool` de LangChain
> - Brancher un modèle Ollama local via `ChatOllama`
> - Observer la boucle ReAct grâce à `verbose=True`
> - Ajouter une mémoire conversationnelle avec `ConversationBufferMemory`

---

## Prérequis techniques

```bash
# Installation des librairies Python
pip install langchain langchain-community langchain-ollama langgraph pandas plotly

# Ollama doit être installé et lancé localement
# Téléchargement : https://ollama.com
# Lancement du service : ollama serve
# Téléchargement du modèle : ollama pull llama3.2
```

**Dataset utilisé :** `job_postings.csv` — 25 114 offres d'emploi, 17 colonnes (titre, localisation, salaire, industrie, etc.)

---
# PARTIE 1 — COURS THÉORIQUE
---

---
## 1. L'IA agentique — de quoi parle-t-on ?

### IA classique vs IA agentique

Un LLM classique fonctionne en **mode réactif** : il reçoit un prompt, génère une réponse, s'arrête.

```
Utilisateur → [LLM] → Réponse   (une seule passe)
```

Un **agent** fonctionne en **mode actif** : il planifie, agit, observe les résultats, se corrige, et itère jusqu'à atteindre l'objectif.

```
Objectif → [LLM décide] → Action → Observation → [LLM réévalue] → Action → ... → Résultat
```

### Ce qui rend un système "agentique"

| Capacité | Description |
|---|---|
| **Planification** | Décomposer un objectif complexe en sous-tâches |
| **Action** | Appeler des tools (fonctions, APIs, bases de données) |
| **Observation** | Lire le résultat de chaque action |
| **Auto-correction** | Modifier la stratégie si l'action échoue |
| **Mémoire** | Conserver le contexte entre les étapes |

> **Exemple concret** : "Analyse ce dataset et génère 3 graphiques" → l'agent lit les colonnes, choisit quels graphiques sont pertinents, les génère, vérifie qu'ils ont bien été créés.

---
## 2. Fonctionnement d'un agent — la boucle ReAct

Le paradigme dominant est **ReAct** (Reason + Act), introduit en 2022 par Google :

```
┌─────────────────────────────────────────────────────────┐
│                    Boucle ReAct                         │
│                                                         │
│  [Observation/Question]                                 │
│         ↓                                               │
│  [Thought] — le LLM raisonne : "Je dois d'abord..."    │
│         ↓                                               │
│  [Action] — le LLM choisit un tool et ses arguments    │
│         ↓                                               │
│  [Observation] — résultat du tool                      │
│         ↓                                               │
│  [Thought] — le LLM analyse le résultat                │
│         ↓                                               │
│  [Action ou Final Answer]                              │
└─────────────────────────────────────────────────────────┘
```

### Les 3 composants fondamentaux

**1. Le LLM (cerveau)**
- Lit l'observation courante + tout l'historique
- Décide : appeler un tool *ou* donner la réponse finale
- Ne fait **pas** d'action lui-même — il délègue

**2. Les Tools (bras)**
- Fonctions Python que l'agent peut déclencher
- Exemples : lire un fichier, faire un calcul, appeler une API, générer un graphique
- Chaque tool retourne une **observation** textuelle

**3. L'Executor (chef d'orchestre)**
- Reçoit la décision du LLM
- Exécute le tool
- Renvoie l'observation au LLM
- Arrête la boucle quand le LLM produit une réponse finale

---
## 3. Types d'agents

### Selon la stratégie de raisonnement

| Type | Principe | Cas d'usage |
|---|---|---|
| **ReAct** | Alternance Thought → Action → Observation | Tâches séquentielles avec tools |
| **Tool Calling** | Le LLM génère directement un appel de fonction structuré | APIs, bases de données (plus fiable que ReAct pur) |
| **Planning** | Planification explicite avant exécution (Plan → Execute) | Tâches longues et complexes |
| **Code** | Le LLM génère du code Python exécutable | Calculs, transformations de données |
| **Multi-agent** | Plusieurs agents spécialisés coordonnés par un orchestrateur | Systèmes complexes (ex: agent "analyste" + agent "visualiseur") |

### Tool Calling Agent — le type utilisé dans ce TP

Le **Tool Calling Agent** est la variante la plus fiable avec des LLMs modernes. Au lieu de générer du texte libre avec des marqueurs `Action:` / `Observation:`, le LLM génère directement un **appel de fonction structuré** (JSON) :

```json
{
  "name": "top_values",
  "arguments": {"filepath": "job_postings.csv", "column": "Company Industry"}
}
```

Avantages :
- Format structuré → moins d'erreurs de parsing
- Compatible avec les modèles supportant le *function calling* (GPT-4, Llama 3.2, Mistral...)
- Disponible nativement dans LangChain via `create_tool_calling_agent`

---
## 4. Orchestration et mémoire

### Orchestration : comment les agents sont coordonnés

**Agent simple (ce TP)**
Un seul agent, une boucle, des tools. Adapté à la majorité des cas.

**Multi-agent (LangGraph)**
Plusieurs agents spécialisés qui se passent des messages. Ex :
```
Superviseur → Agent Analyse → Agent Visualisation → Agent Rapport
```

**Séquentiel vs parallèle**
- Séquentiel : chaque step attend le précédent (par défaut)
- Parallèle : plusieurs tools en simultané (LangGraph, agents avancés)

### Mémoire : ce que l'agent "sait"

| Type | Durée | Implémentation |
|---|---|---|
| **Mémoire de contexte** | Durée de la boucle (quelques steps) | Scratchpad dans le prompt |
| **Mémoire de session** | Durée de la conversation | `ConversationBufferMemory` |
| **Mémoire long terme** | Persistante entre sessions | Base de données, fichier JSON |
| **Mémoire sémantique** | Recherche par similarité | Vector store (Chroma, FAISS) |

> Dans ce TP, nous utilisons `ConversationBufferMemory` (mémoire de session) en Partie 4.

---
## 5. LangChain — présentation et écosystème

### Qu'est-ce que LangChain ?

LangChain est un **framework open-source** (Python/JS) pour construire des applications basées sur des LLMs. Créé en 2022, il est devenu le standard de facto pour les applications LLM en production.

**Philosophie** : composer des briques (LLM, tools, mémoire, prompt) en pipelines appelés *chains* ou *agents*.

### Les composants clés

| Composant | Rôle | Ce que vous allez utiliser |
|---|---|---|
| **LLMs / Chat Models** | Interface vers les modèles | `ChatOllama` (local) |
| **Prompt Templates** | Structurer les messages | `ChatPromptTemplate` |
| **Tools** | Actions de l'agent | `@tool` decorator |
| **Agents** | Cerveau décisionnel | `create_tool_calling_agent` |
| **Executor** | Boucle d'exécution | `AgentExecutor` |
| **Memory** | Historique conversationnel | `ConversationBufferMemory` |

### L'écosystème LangChain

```
LangChain Core          — briques de base (LLM, prompts, tools, chains)
LangChain Community     — intégrations (Ollama, OpenAI, HuggingFace, Chroma...)
LangGraph               — agents multi-étapes avec état (le futur des agents)
LangSmith               — observabilité, debugging, évaluation des traces
LangServe               — déploiement d'agents en API REST
```

### Pourquoi LangChain pour ce TP ?

- **`verbose=True`** — voir chaque step sans code supplémentaire
- **Mémoire native** — `ConversationBufferMemory` en 3 lignes
- **Prompt explicite** — vous contrôlez exactement ce que voit le LLM
- **Modèles locaux** — `ChatOllama` branché sur votre installation Ollama

## Architecture d'un agent LangGraph

`create_react_agent` est la fonction standard de **LangGraph** pour créer un agent ReAct :

```python
agent = create_react_agent(llm, tools)
result = agent.invoke({"messages": [("human", "votre question")]})
print(result["messages"][-1].content)  # réponse finale
```

Le flux interne à chaque invocation :

```
messages → [LLM décide] ──── si tool call ──→ [Tool exécuté] ──→ [LLM réévalue]
                         └─── si réponse finale ──→ résultat
```

| Objet | Rôle | Classe |
|---|---|---|
| **LLM** | Cerveau — décide quel tool appeler | `ChatOllama` |
| **Tools** | Actions disponibles | `@tool` |
| **Agent** | Boucle ReAct complète | `create_react_agent` |
| **MemorySaver** | Mémoire entre appels | `MemorySaver` + `thread_id` |

In [ ]:
# NE PAS MODIFIER — Vérification de l'environnement
import importlib

packages = {
    'langchain_core': 'langchain-core',
    'langchain_ollama': 'langchain-ollama',
    'langgraph': 'langgraph',
    'pandas': 'pandas',
    'plotly': 'plotly',
}

all_ok = True
for module, pkg in packages.items():
    try:
        importlib.import_module(module)
        print(f"✓ {pkg}")
    except ImportError:
        print(f"✗ {pkg} manquant  →  pip install {pkg}")
        all_ok = False

if all_ok:
    print("\n✓ Environnement prêt")
else:
    print("\n✗ Installez les packages manquants puis redémarrez le kernel")

In [ ]:
# NE PAS MODIFIER — Imports communs
import pandas as pd
import plotly.express as px
import plotly.io as pio
import webbrowser, json, re, os
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)  # LangGraph v1.0 migration warnings

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI          # meilleure compatibilité tool calling
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

pio.renderers.default = 'browser'
print("✓ Imports OK")

---
# PARTIE 2 — TRAVAUX PRATIQUES
---

---
## Exercice 1 — Construire les tools : les "mains" de l'agent

**Objectif de ce TP** : à la fin, vous aurez un système qui prend n'importe quelle phrase en langage naturel et génère automatiquement un dashboard HTML avec 2 graphiques.

Pour y arriver, on construit les briques une par une :
- Partie 1 → les tools (fonctions que l'agent peut appeler)
- Partie 2 → premier agent (LLM + 1 tool → graphique)
- Partie 3 → agent multi-tools (analyse + graphique en une requête)
- Partie 4 → mémoire (l'agent se souvient du contexte)
- Partie 5 → dashboard (2 graphiques dans un seul HTML)
- Partie 6 → projet final (langage naturel → dashboard complet)

In [ ]:
# NE PAS MODIFIER — Étape 1A : tools de base (exploration + graphique)

import re, glob
import pandas as pd

# ── Helper partagé : fuzzy matching robuste ──────────────────────────────────
def fuzzy_col(df, col_name: str) -> str:
    """Trouve la vraie colonne dans df à partir d'un nom approximatif."""
    col_clean = re.sub(r'[^a-zA-Z0-9 ]', '', col_name).strip().lower()
    for c in df.columns:
        if c.lower() == col_clean:
            return c
    for c in df.columns:
        if col_clean in c.lower() or c.lower() in col_clean:
            return c
    words = [w for w in col_clean.split() if len(w) >= 4]
    for word in words:
        stems = [word,
                 word[:-1] if len(word) > 4 else '',
                 word[:-2] if len(word) > 5 else '']
        for c in df.columns:
            if any(s and s in c.lower() for s in stems):
                return c
    return col_name

def _col_not_found_msg(df, col_name, kind='catégorielle'):
    if kind == 'catégorielle':
        available = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    else:
        available = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    return (f"Colonne '{col_name}' introuvable. "
            f"Utilise un de ces noms exacts : {available}")

def _clean_filepath(filepath: str) -> str:
    """Sanitise le filepath : JSON array, faux chemin absolu, typo de nom."""
    fp = filepath.strip()
    # Cas 1 : tableau JSON  ["file.csv"]
    if fp.startswith("["):
        fp = fp.strip("[]").split(",")[0].strip()
        fp = fp.strip('"').strip("'")
    # Cas 2 : faux chemin absolu inventé par le LLM (/path/to/file.csv)
    if not os.path.exists(fp) and "/" in fp:
        basename = fp.split("/")[-1]
        if os.path.exists(basename):
            fp = basename
    # Cas 3 : typo dans le nom (job_posting.csv → job_postings.csv)
    if not os.path.exists(fp):
        name = os.path.basename(fp)
        candidates = glob.glob("*.csv")
        # prefer exact prefix match
        for c in candidates:
            if c.startswith(name.split(".")[0][:8]):
                fp = c
                break
        else:
            if len(candidates) == 1:
                fp = candidates[0]
    return fp

@tool
def describe_dataset(filepath: str = "job_postings.csv") -> str:
    """Charge un CSV et retourne ses dimensions et colonnes. Utiliser EN PREMIER.
    Args:
        filepath: Chemin vers le fichier CSV à analyser.
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable. Utilise 'job_postings.csv'."
    cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    return (f"{df.shape[0]} lignes, {df.shape[1]} colonnes.\n"
            f"Catégorielles (utilise bar_chart, noms exacts) : {cat_cols}\n"
            f"Numériques (utilise numeric_stats, noms exacts) : {num_cols}")

@tool
def bar_chart(filepath: str = "job_postings.csv", column: str = "", title: str = '') -> str:
    """Génère un bar chart horizontal des 10 valeurs les plus fréquentes. Sauvegarde en HTML.
    Args:
        filepath: Chemin vers le fichier CSV.
        column: Nom exact ou partiel de la colonne catégorielle.
        title: Titre du graphique (optionnel).
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable. Utilise 'job_postings.csv'."
    col = fuzzy_col(df, column)
    if col not in df.columns:
        return _col_not_found_msg(df, column, 'catégorielle')
    top = df[col].value_counts().head(10).reset_index()
    top.columns = [col, 'count']
    fig = px.bar(top.sort_values('count'), x='count', y=col, orientation='h', title=title or col)
    out = f'chart_{col.replace(" ", "_")}.html'
    fig.write_html(out)
    webbrowser.open(f'file://{os.path.abspath(out)}')
    return f"Graphique sauvegardé : {out}"

# ── Tests directs ─────────────────────────────────────────────────────────────
print("=== describe_dataset ===")
print(describe_dataset.invoke({"filepath": "job_postings.csv"}))
print("\n=== bar_chart ===")
print(bar_chart.invoke({"filepath": "job_postings.csv",
                         "column": "Company Industry",
                         "title": "Top 10 industries"}))


### Exercice 1 — Définir top_values et numeric_stats

Ces deux tools complètent la boîte à outils de l'agent :
- `top_values` : compte les occurrences d'une valeur catégorielle (sans graphique)
- `numeric_stats` : calcule min/max/moyenne/médiane d'une colonne numérique

**Modèle** : même structure que `bar_chart` — `@tool`, docstring avec `Args:`, fuzzy matching.

In [ ]:
# NE PAS MODIFIER — Démonstration : modèle à reproduire pour l'exercice

@tool
def top_values(filepath: str = "job_postings.csv", column: str = "") -> str:
    """Retourne les 10 valeurs les plus fréquentes d'une colonne catégorielle (texte).
    Args:
        filepath: Chemin vers le fichier CSV.
        column: Nom exact ou partiel de la colonne catégorielle.
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable. Utilise 'job_postings.csv'."
    col = fuzzy_col(df, column)
    if col not in df.columns:
        return _col_not_found_msg(df, column, 'catégorielle')
    result = df[col].value_counts().head(10)
    return f"Top 10 — {col} :\n" + result.to_string()

print(top_values.invoke({"filepath": "job_postings.csv", "column": "Company Industry"}))


In [ ]:
# EXERCICE 1B — Définir numeric_stats
# Retourne min, max, moyenne, médiane d'une colonne numérique.
# Inspirez-vous de top_values ci-dessus : même structure, même fuzzy_col.

@tool
def numeric_stats(filepath: str, column: str) -> str:
    """TODO : compléter la docstring
    Args:
        filepath: TODO
        column: TODO
    """
    # TODO : lire le CSV, trouver la colonne avec fuzzy_col, calculer les stats
    # Astuce : utilisez .dropna() pour ignorer les valeurs manquantes
    pass

# Test
print(numeric_stats.invoke({"filepath": "job_postings.csv", "column": "Minimum Pay"}))

---
## Exercice 2 — Premier agent : langage naturel → graphique

On a maintenant des tools qui génèrent des graphiques HTML. L'étape suivante : brancher un LLM qui **décide tout seul** quel tool appeler en réponse à une phrase en langage naturel.

Schéma :
```
"Génère un bar chart pour Company Industry" → [Agent] → bar_chart() → chart_Company_Industry.html
```

La fonction `show_steps()` permet de voir exactement ce que l'agent fait à chaque étape.

In [ ]:
# NE PAS MODIFIER — Étape 2A : show_steps — voir l'agent en action

# Mots-clés qui indiquent qu'un tool a sauvegardé un fichier.
# Si le tool a réussi, on affiche son résultat plutôt que la réponse du LLM
# (llama3.2 hallucine souvent des tableaux/stats après un tool qui génère un fichier).
_SAVED_KEYWORDS = ("sauvegardé", "saved", "Dashboard sauvegardé", "Graphique sauvegardé")

def show_steps(agent, question: str, config: dict = None):
    """Lance l'agent et affiche chaque etape en temps reel."""
    print(f"► Question : {question}")
    print("─" * 60)

    final = ""
    last_tool_result = ""   # dernier resultat de tool (utilise si le LLM hallucine)

    for chunk in agent.stream({"messages": [("human", question)]}, config or {}):

        if "agent" in chunk:
            msg = chunk["agent"]["messages"][0]
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  ▶ Appel tool : {tc['name']}")
                    print(f"    Arguments  : {tc['args']}")
            else:
                final = msg.content

        elif "tools" in chunk:
            obs = chunk["tools"]["messages"][0].content
            print(f"  ◀ Résultat  : {obs[:150]}")
            if any(kw in obs for kw in _SAVED_KEYWORDS):
                last_tool_result = obs.split("\n")[0]

    print("─" * 60)

    # Si le tool a sauvegarde un fichier, ignorer la reponse du LLM
    # (evite les tableaux/stats hallucines que llama3.2 genere apres make_dashboard/bar_chart)
    if last_tool_result and any(kw in last_tool_result for kw in _SAVED_KEYWORDS):
        print(f"✓ {last_tool_result}")
    else:
        print(f"✓ Réponse finale : {final[:200]}")

    return final

print("✓ show_steps() prêt")


In [ ]:
# NE PAS MODIFIER — Étape 2B : premier agent NL → graphique

# ── 1. Créer le LLM ──────────────────────────────────────────────────────────
# ChatOpenAI pointe sur l'API Ollama en mode compatibilité OpenAI (port 11434/v1)
# Plus fiable que ChatOllama pour le tool calling avec llama3.2
llm = ChatOpenAI(
    model="llama3.2",
    base_url="http://localhost:11434/v1",
    api_key="ollama"               # clé fictive requise par l'API
)

# ── 2. Créer l'agent ─────────────────────────────────────────────────────────
# System prompt : évite que llama3.2 hallucine du HTML après l'exécution du tool.
# Règle universelle pour ce TP : quand un tool confirme qu'un fichier est sauvegardé,
# répondre avec une seule phrase — jamais de HTML, SVG ou lien <a href>.
_DEMO_PROMPT = (
    "You are a data visualization assistant working with job_postings.csv. "
    "Always call tools directly. "
    "When a tool reports that a file has been saved, reply with ONE sentence only. "
    "Never write HTML, SVG, or hyperlinks in your response. After a tool call succeeds, output ONLY: 'File saved: <filename>.' Do NOT add any analysis, summary, statistics, or description. Stop immediately after the confirmation sentence."
)
agent_demo = create_react_agent(llm, [describe_dataset, bar_chart], prompt=_DEMO_PROMPT)

# ── 3. Lancer en langage naturel ─────────────────────────────────────────────
show_steps(
    agent_demo,
    "Explore job_postings.csv puis génère un bar chart pour la colonne 'Company Industry'."
)

### Exercice 2 — Autre graphique en langage naturel

Même agent (`agent_demo`), autre demande. Essayez de demander un graphique pour une **autre colonne catégorielle**.

**Colonnes catégorielles disponibles** : `Job Title`, `Job Position Type`, `Job Position Level`, `Job Skills`, `Job Location`, `Company Name`, `Company Industry`, `Company Size`

> **Conseil** : incluez le nom exact de la colonne dans votre question pour guider le modèle.

In [ ]:
# EXERCICE 2 — Demander un graphique pour une autre colonne
# Modifiez la question pour obtenir un bar chart sur une colonne DIFFÉRENTE
# de "Company Industry" (ex : Job Position Type, Work Type, Formatted Experience Level...)

show_steps(
    agent_demo,
    "TODO : écrivez votre question en langage naturel ici"  # modifier cette ligne
)

---
## Exercice 3 — Agent multi-tools : analyse + graphique en une seule requête

Avec les 4 tools (`describe_dataset`, `top_values`, `numeric_stats`, `bar_chart`), l'agent peut répondre à des demandes complexes en appelant plusieurs tools successivement :

```
"Stats sur Minimum Pay et bar chart pour Job Position Type"
→ Agent appelle numeric_stats, puis bar_chart → 2 résultats en 1 requête
```

In [ ]:
# NE PAS MODIFIER — Étape 3A : agent avec 4 tools

# Tous les tools définis en Partie 1 sont disponibles
all_tools = [describe_dataset, top_values, numeric_stats, bar_chart]

# ── Créer l'agent multi-tools ────────────────────────────────────────────────
_MULTI_PROMPT = (
    "You are a data analysis assistant working with job_postings.csv. "
    "Always call tools directly. "
    "When a tool reports that a file has been saved, reply with ONE sentence only. "
    "Never write HTML, SVG, or hyperlinks in your response. After a tool call succeeds, output ONLY: 'File saved: <filename>.' Do NOT add any analysis, summary, statistics, or description. Stop immediately after the confirmation sentence."
)
agent_multi = create_react_agent(llm, all_tools, prompt=_MULTI_PROMPT)

# ── Requête combinée : plusieurs tools en séquence ───────────────────────────
# On cite les noms de colonnes exacts pour éviter les hallucinations du LLM
show_steps(
    agent_multi,
    "Dans job_postings.csv, calcule les statistiques de 'Minimum Pay' "
    "ET génère un bar chart pour 'Company Industry'."
)

### Exercice 3 — Requête multi-étapes

Formulez une question qui nécessite **au moins 2 tools différents** :
1. Un tool d'analyse textuelle (`top_values` ou `numeric_stats`)
2. Un tool de visualisation (`bar_chart`)

Vérifiez dans les steps que les 2 tools ont bien été appelés.

In [ ]:
# NE PAS MODIFIER — Étape 3B : requête multi-étapes

show_steps(
    agent_multi,
    "Analyse job_postings.csv : stats sur 'Maximum Pay' "
    "et bar chart pour 'Job Position Level'."
)

---
## Exercice 4 — Mémoire : l'agent se souvient du contexte

Sans mémoire, chaque appel à l'agent repart de zéro. Avec `MemorySaver`, l'agent peut s'appuyer sur les échanges précédents pour répondre à des questions de suivi.

**Cas d'usage** : demander d'abord une exploration, puis demander un graphique basé sur ce que l'agent vient de voir — sans répéter les colonnes.

In [ ]:
# NE PAS MODIFIER — Étape 4A : agent avec mémoire → questions liées

# ── Mémoire : MemorySaver stocke l'historique en RAM ─────────────────────────
memory = MemorySaver()

# ── thread_id : identifiant de session ───────────────────────────────────────
# Même thread_id = même conversation (l'agent se souvient)
# thread_id différent = nouvelle conversation indépendante
config = {"configurable": {"thread_id": "analyse_s9"}}

_MEM_PROMPT = (
    "You are a data analysis assistant working with job_postings.csv. "
    "To explore a dataset, ALWAYS call describe_dataset — never top_values. "
    "Always call tools directly. "
    "After a tool call succeeds, output ONLY: 'File saved: <filename>.' "
    "Do NOT add any analysis, summary, statistics, or description. "
    "Stop immediately after the confirmation sentence."
)
agent_mem = create_react_agent(
    llm,
    [describe_dataset, top_values, numeric_stats, bar_chart],
    checkpointer=memory,   # branche la mémoire sur l'agent
    prompt=_MEM_PROMPT
)

# ── Question 1 : exploration ──────────────────────────────────────────────────
# On précise l'outil à utiliser pour éviter que llama3.2 appelle top_values à la place
print("=== Question 1 : exploration ===")
show_steps(
    agent_mem,
    "Utilise describe_dataset sur job_postings.csv et liste les colonnes catégorielles.",
    config
)

# ── Question 2 : graphique basé sur Q1 ───────────────────────────────────────
# L'agent connaît les colonnes depuis Q1 — on cite une colonne catégorielle valide
print("\n=== Question 2 : graphique basé sur la Q1 ===")
show_steps(
    agent_mem,
    "Génère un bar chart pour la colonne 'Company Industry' que tu viens de voir.",
    config
)

### Exercice 4 — Conversation en deux temps avec graphique

Réutilisez `agent_mem` avec un **nouveau thread_id** (pour une conversation fraîche) et posez deux questions liées :
1. Une exploration du dataset
2. Une demande de graphique basée sur ce que l'agent a trouvé

Changez le `thread_id` pour éviter de mélanger avec la conversation précédente.

In [ ]:
# NE PAS MODIFIER — Étape 4B : conversation mémoire

config_ex4 = {"configurable": {"thread_id": "exercice_4"}}

# Question 1 : exploration — l'agent appelle top_values sur Job Position Type
print("=== Q1 : quels types de postes existent ? ===")
show_steps(
    agent_mem,
    "Utilise top_values sur la colonne 'Job Position Type' de job_postings.csv "
    "et liste les valeurs les plus fréquentes.",
    config_ex4
)

# Question 2 : graphique — on cite la colonne explicitement car llama3.2 (3B)
# ne déduit pas fiablement "la colonne que tu viens de mentionner" depuis la mémoire.
print("\n=== Q2 : bar chart de Job Position Type ===")
show_steps(
    agent_mem,
    "Génère un bar chart pour la colonne 'Job Position Type' depuis job_postings.csv.",
    config_ex4
)

---
## Exercice 5 — Le dashboard : deux graphiques dans un seul HTML

Jusqu'ici l'agent générait un graphique à la fois. Pour le projet final, on veut un **dashboard HTML** avec 2 graphiques côte à côte, généré en une seule phrase.

On crée un tool `make_dashboard` qui prend 2 colonnes et génère un fichier HTML avec `plotly.subplots`.

In [ ]:
# NE PAS MODIFIER — Étape 5A : tool make_dashboard — 2 graphiques dans un HTML

from plotly.subplots import make_subplots
import plotly.graph_objects as go

@tool
def make_dashboard(filepath: str, column1: str, column2: str, title: str) -> str:
    """Génère un dashboard HTML avec 2 bar charts côte à côte. Sauvegarde et ouvre dans le navigateur.
    IMPORTANT : utilise uniquement des noms de colonnes retournés par describe_dataset.
    Args:
        filepath: Chemin vers le fichier CSV.
        column1: Première colonne catégorielle (nom exact ou partiel).
        column2: Deuxième colonne catégorielle (nom exact ou partiel).
        title: Titre du dashboard.
    """
    df = pd.read_csv(_clean_filepath(filepath))

    # Normaliser et vérifier les deux colonnes
    col1 = fuzzy_col(df, column1)
    col2 = fuzzy_col(df, column2)

    # Si une colonne est introuvable → retourner la liste des colonnes disponibles
    # (le LLM lira ce message et relancera avec les bons noms)
    if col1 not in df.columns or col2 not in df.columns:
        cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        missing = [c for c, real in [(column1, col1), (column2, col2)] if real not in df.columns]
        return (f"Colonnes introuvables : {missing}. "
                f"Utilise ces noms exacts : {cat_cols}")

    # Préparer les données : top 10 pour chaque colonne
    def top10(col):
        t = df[col].value_counts().head(10).reset_index()
        t.columns = [col, 'count']
        return t.sort_values('count')

    # Créer la figure avec 2 sous-graphiques côte à côte
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=[col1, col2],
                        horizontal_spacing=0.15)
    t1 = top10(col1)
    fig.add_trace(go.Bar(x=t1['count'], y=t1[col1], orientation='h', name=col1), row=1, col=1)
    t2 = top10(col2)
    fig.add_trace(go.Bar(x=t2['count'], y=t2[col2], orientation='h', name=col2), row=1, col=2)
    fig.update_layout(title_text=title, height=500, showlegend=False)

    out = 'dashboard.html'
    fig.write_html(out)
    webbrowser.open(f'file://{os.path.abspath(out)}')
    return f"Dashboard sauvegardé : {out} ({col1} + {col2})"

# Test direct
print(make_dashboard.invoke({
    "filepath": "job_postings.csv",
    "column1": "Company Industry",
    "column2": "Job Position Type",
    "title": "Analyse des offres d'emploi"
}))

In [ ]:
# NE PAS MODIFIER — Étape 5B : agent NL → dashboard

_DASH_PROMPT = (
    "You are a data visualization assistant working with job_postings.csv. "
    "CRITICAL RULES: "
    "1. Always call tools directly — never describe a tool call as text. "
    "2. Use EXACT column names as given by the user — NEVER abbreviate or shorten them. "
    "   Example: use 'Company Industry', NOT 'Company'. Use 'Job Position Type', NOT 'Job Type'. "
    "3. After any tool call, your ENTIRE response must be exactly: "
    "   'Dashboard generated: dashboard.html' — nothing else. "
    "4. Do NOT write tables, lists, statistics, HTML, or any description. Stop immediately."
)

# L'agent dispose de describe_dataset pour explorer + make_dashboard pour générer
agent_dash = create_react_agent(llm, [describe_dataset, make_dashboard], prompt=_DASH_PROMPT)

# Les noms de colonnes sont cités explicitement pour guider llama3.2
show_steps(
    agent_dash,
    "Appelle make_dashboard avec column1='Company Industry', column2='Job Position Type', "
    "filepath='job_postings.csv', title='Analyse emploi'."
)

### Exercice 5 — Dashboard sur d'autres colonnes

Demandez à `agent_dash` de créer un dashboard avec **deux colonnes différentes** de la démonstration.

**Colonnes catégorielles disponibles** : `Job Position Type`, `Job Position Level`, `Job Skills`, `Job Location`, `Company Name`, `Company Industry`, `Company Size`

> Suggestion : `Job Position Level` + `Company Size`

In [ ]:
# NE PAS MODIFIER — Étape 5C : dashboard sur d'autres colonnes

show_steps(
    agent_dash,
    "Appelle make_dashboard avec column1='Job Position Type', column2='Job Position Level', "
    "filepath='job_postings.csv', title='Types et niveaux de postes'."
)

---
## Projet final : système NL → dashboard complet

Vous avez maintenant tous les composants :
- ✅ Tools : `describe_dataset`, `top_values`, `numeric_stats`, `bar_chart`, `make_dashboard`
- ✅ Agent multi-tools : `create_react_agent`
- ✅ Mémoire : `MemorySaver` + `thread_id`
- ✅ Streaming : `show_steps`

**Objectif** : construire un agent qui reçoit n'importe quelle demande en langage naturel et génère automatiquement le dashboard le plus approprié.

In [ ]:
# EXERCICE FINAL — Système NL → dashboard complet
#
# Construire un agent avec TOUS les tools + mémoire, capable de :
# 1. Explorer le dataset (describe_dataset)
# 2. Analyser des colonnes (top_values, numeric_stats)
# 3. Générer un graphique simple (bar_chart) ou un dashboard (make_dashboard)
#
# Étapes à compléter :
# TODO 1 : définir all_tools_final avec les 5 tools
# TODO 2 : instancier memory_final = MemorySaver()
# TODO 3 : créer agent_final = create_react_agent(llm, all_tools_final,
#           checkpointer=memory_final,
#           prompt="You are a data analysis assistant. Always call tools directly.")
# TODO 4 : définir config_final avec un thread_id unique
# TODO 5 : lancer show_steps(agent_final, ..., config_final) sur chaque requête

all_tools_final = [...]   # TODO 1
memory_final = ...        # TODO 2
agent_final  = ...        # TODO 3
config_final = ...        # TODO 4

# Requête 1 : exploration (lancer en premier)
requete_exploration = (
    "Utilise describe_dataset sur job_postings.csv et retourne "
    "la liste exacte des colonnes catégorielles disponibles."
)

# Requête 2 : génération du dashboard (lancer après avoir les colonnes)
requete_dashboard = (
    "Génère un dashboard avec make_dashboard en utilisant "
    "'Company Industry' et 'Job Position Type' depuis job_postings.csv. "
    "Titre : 'Analyse des offres d emploi'"
)

# TODO 5 : lancer les deux requêtes avec show_steps
# show_steps(agent_final, requete_exploration, config_final)
# show_steps(agent_final, requete_dashboard,   config_final)

---
## Exercice 7 — System prompt : contrôler le comportement de l'agent

Jusqu'ici nos agents n'avaient **aucun system prompt** : le LLM faisait ce qu'il voulait.
Un **system prompt** est un message système injecté avant chaque échange — il définit le rôle, les règles, le style de l'agent.

Avec `create_react_agent`, il suffit d'un paramètre `prompt=` :

```python
agent = create_react_agent(llm, tools, prompt="Tu es un analyste RH strict...")
```

On repart de l'agent du projet final (`llm`, `describe_dataset`, `make_dashboard`) pour comparer le comportement avec et sans prompt.

In [ ]:
# NE PAS MODIFIER — Étape 7A : comparer sans et avec system prompt

# ── Agent SANS prompt (comportement par défaut) ───────────────────────────────
print("=== SANS system prompt ===")
agent_no_prompt = create_react_agent(llm, [describe_dataset, make_dashboard])
show_steps(agent_no_prompt, "Fais une analyse rapide de job_postings.csv")

# ── Définir un system prompt explicite ───────────────────────────────────────
SYSTEM_PROMPT = """Tu es un analyste de données RH spécialisé travaillant avec job_postings.csv.
Règles absolues :
- Commence TOUJOURS par appeler describe_dataset avant de générer un graphique.
- Réponds en français, de façon concise (2-3 phrases max).
- Si la demande est floue, demande une clarification plutôt que d'inventer.
- N'écris JAMAIS de code Python, matplotlib, HTML ou SVG dans ta réponse.
- Après un appel de tool réussi, réponds avec UNE seule phrase. Arrête-toi immédiatement."""
print("\n=== AVEC system prompt ===")
agent_v2 = create_react_agent(llm, [describe_dataset, make_dashboard], prompt=SYSTEM_PROMPT)
show_steps(agent_v2, "Fais une analyse rapide de job_postings.csv")

In [ ]:
# NE PAS MODIFIER — Étape 7B : temperature — déterminisme vs créativité

# temperature=0   → réponses identiques à chaque run  (recommandé pour les agents)
# temperature=0.8 → réponses variables                 (adapté aux tâches créatives)

llm_t0 = ChatOpenAI(model="llama3.2", base_url="http://localhost:11434/v1",
                     api_key="ollama", temperature=0)
llm_t8 = ChatOpenAI(model="llama3.2", base_url="http://localhost:11434/v1",
                     api_key="ollama", temperature=0.8)

question = "Quels types de postes sont les plus nombreux dans job_postings.csv ?"

print("=== temperature=0 (déterministe) ===")
agent_t0 = create_react_agent(llm_t0, [describe_dataset], prompt=SYSTEM_PROMPT)
show_steps(agent_t0, question)

print("\n=== temperature=0.8 (variable) — relancez plusieurs fois pour voir la variance ===")
agent_t8 = create_react_agent(llm_t8, [describe_dataset], prompt=SYSTEM_PROMPT)
show_steps(agent_t8, question)

### Exercice 6 — Écrire votre propre system prompt

Créez un agent "analyste strict" qui respecte ces règles :
- Ne génère un graphique **que** si l'utilisateur mentionne explicitement un nom de colonne
- Répond **toujours en anglais**
- Commence chaque réponse par appeler `describe_dataset`

Testez avec deux questions : une vague ("Show me something") et une précise ("Show Company Industry distribution").
Observez que seule la 2ème déclenche un graphique.

In [ ]:
# EXERCICE 6 — Écrire votre propre system prompt
#
# TODO 1 : écrire MY_PROMPT — agent analyste strict (voir consignes ci-dessus)
#           IMPORTANT : précisez le nom du fichier 'job_postings.csv' dans le prompt,
#           sinon le LLM peut inventer un chemin fictif.
# TODO 2 : créer agent_ex6 = create_react_agent(llm, [...], prompt=MY_PROMPT)
# TODO 3 : tester avec les deux questions ci-dessous

MY_PROMPT = """..."""\  # TODO 1

agent_ex6 = None  # TODO 2

show_steps(agent_ex6, "Show me something about the jobs")         # → doit demander une clarification
show_steps(agent_ex6, "Show Company Industry distribution")       # → doit générer un graphique

---
## Exercice 8 — Human-in-the-Loop (HITL) : valider avant d'agir

Par défaut un agent exécute ses tools **sans demander confirmation**.
Le **Human-in-the-Loop** (HITL) ajoute un **point d'arrêt** entre la décision du LLM et l'exécution du tool.

```python
agent = create_react_agent(llm, tools, checkpointer=memory,
                            interrupt_before=["tools"])  # ← point d'arrêt
```

Quand l'agent veut appeler un tool, il s'arrête. On peut alors :
1. **Valider** → `agent.invoke(None, config)` — le tool s'exécute
2. **Modifier** les arguments → `agent.update_state(config, ...)` puis `invoke(None, config)`
3. **Annuler** → ne pas reprendre

On repart du même LLM et des mêmes tools que le projet final.

In [ ]:
# NE PAS MODIFIER — Étape 8A : HITL — inspecter avant d'exécuter le tool

from langchain_core.messages import AIMessage

# System prompt : évite que le LLM génère du HTML/SVG après l'exécution du tool.
_HITL_PROMPT = (
    "You are a data visualization assistant working with job_postings.csv. "
    "Always call tools directly. "
    "When a tool reports that a file has been saved, reply with ONE sentence only: "
    "'Chart saved: <filename>'. Never write HTML or SVG code in your response. After a tool call succeeds, output ONLY: 'File saved: <filename>.' Do NOT add any analysis, summary, statistics, or description. Stop immediately after the confirmation sentence."
)

memory_hitl = MemorySaver()
agent_hitl  = create_react_agent(
    llm,
    [describe_dataset, bar_chart, make_dashboard],
    checkpointer=memory_hitl,
    interrupt_before=["tools"],    # ← l'agent s'arrête avant chaque tool call
    prompt=_HITL_PROMPT
)
config_hitl = {"configurable": {"thread_id": "hitl_demo"}}

# Lancer l'agent — il va s'arrêter AVANT d'appeler son premier tool
print("Lancement de l'agent...")
for chunk in agent_hitl.stream(
    {"messages": [("human", "Génère un bar chart de 'Company Industry' depuis job_postings.csv")]},
    config_hitl
):
    if "agent" in chunk:
        msg = chunk["agent"]["messages"][0]
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            tc = msg.tool_calls[0]
            print(f"\n⏸  Agent pausé AVANT d'appeler : {tc['name']}")
            print(f"   Arguments prévus : {tc['args']}")

# Inspecter l'état interne
state = agent_hitl.get_state(config_hitl)
print(f"\n   État : next={state.next}  (l'agent attend votre décision)")

In [ ]:
# NE PAS MODIFIER — Étape 8B : HITL — valider et reprendre

# L'agent est pausé depuis l'étape 8A.
# invoke(None, config) signifie : "pas de nouveau message, continue là où tu t'es arrêté".

print("✅ Validation humaine : le tool peut s'exécuter")
result = agent_hitl.invoke(None, config_hitl)

final_msg = result["messages"][-1].content
print(f"\nRéponse finale : {final_msg[:300]}")

In [ ]:
# NE PAS MODIFIER — Étape 8C : HITL — modifier le tool call avant de reprendre

# Cas d'usage : l'agent a choisi la mauvaise colonne → on corrige avant exécution.

config_hitl2 = {"configurable": {"thread_id": "hitl_modify"}}

# Lancer — l'agent s'arrête avant bar_chart (colonne 'salaire' inexistante)
print("Lancement avec une demande ambiguë...")
for chunk in agent_hitl.stream(
    {"messages": [("human", "Génère un bar chart de la colonne salaire depuis job_postings.csv")]},
    config_hitl2
):
    if "agent" in chunk:
        msg = chunk["agent"]["messages"][0]
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            tc = msg.tool_calls[0]
            print(f"⏸  Pausé — tool : {tc['name']}, args : {tc['args']}")

# Récupérer le message AI courant et corriger la colonne
state2  = agent_hitl.get_state(config_hitl2)
last_ai = state2.values["messages"][-1]

corrected = AIMessage(
    content    = last_ai.content,
    tool_calls = [{
        "id":   last_ai.tool_calls[0]["id"],
        "name": last_ai.tool_calls[0]["name"],
        "args": {**last_ai.tool_calls[0]["args"], "column": "Company Industry"}
    }]
)

# as_node="agent" : indique à LangGraph que la correction remplace la sortie
# du nœud "agent" → le graphe reprend directement au nœud "tools" avec
# les arguments corrigés, sans relancer une planification du LLM.
agent_hitl.update_state(config_hitl2, {"messages": [corrected]}, as_node="agent")
print("\n✏️  Colonne corrigée → 'Company Industry'")

# Reprendre — le tool corrigé s'exécute
print("Reprise de l'agent...")
final = ""
for chunk in agent_hitl.stream(None, config_hitl2):
    if "agent" in chunk:
        msg = chunk["agent"]["messages"][0]
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            tc = msg.tool_calls[0]
            print(f"  ▶ Tool : {tc['name']}  args={tc['args']}")
        else:
            final = msg.content
    elif "tools" in chunk:
        print(f"  ◀ Résultat : {chunk['tools']['messages'][0].content[:120]}")

print(f"\n✓ Réponse finale : {final[:200]}")

### Exercice 7 — HITL sur le projet final

Reprenez le projet final (make_dashboard avec `describe_dataset`, `top_values`, `numeric_stats`, `bar_chart`, `make_dashboard`) et ajoutez un point d'arrêt HITL.

**Étapes :**
1. Créer `agent_hitl_final` avec `interrupt_before=["tools"]` et un `MemorySaver`
2. Lancer l'agent avec une demande de dashboard en langage naturel
3. Inspecter l'état : quelle colonne l'agent a-t-il choisie ?
4. Valider (ou modifier) et reprendre

In [ ]:
# EXERCICE 7 — HITL sur le projet final
#
# TODO 1 : créer memory_ex7 = MemorySaver()
# TODO 2 : créer agent_hitl_final avec interrupt_before=["tools"] + tous les tools du projet final
# TODO 3 : lancer l'agent avec une demande de dashboard (en langage naturel)
# TODO 4 : inspecter l'état avec get_state et afficher les arguments du tool call
# TODO 5 : valider avec invoke(None, config_ex7)

config_ex7 = {"configurable": {"thread_id": "hitl_ex7"}}

memory_ex7       = None   # TODO 1
agent_hitl_final = None   # TODO 2

# TODO 3 — lancer l'agent

# TODO 4 — inspecter

# TODO 5 — valider

---
## Synthèse — Ce que vous avez appris

Vous avez construit un **système agentique NL → dashboard** complet avec LangGraph :

| Composant | Rôle | Ce que vous avez codé |
|---|---|---|
| `@tool` | Action exécutable par l'agent | `describe_dataset`, `bar_chart`, `make_dashboard`... |
| `ChatOllama` | LLM local (Ollama) | `ChatOllama(model="llama3.2")` |
| `create_react_agent` | Boucle ReAct | agent qui lit → décide → agit → répète |
| `MemorySaver` | Mémoire de session | historique conservé via `thread_id` |
| `show_steps` | Observabilité | visualisation des tool calls en temps réel |

**Le pattern final** :
```python
agent = create_react_agent(llm, tools, checkpointer=MemorySaver())
show_steps(agent, "Votre question en langage naturel", config)
# → génère automatiquement dashboard.html
```

### Pour aller plus loin
- **Streamlit** : envelopper `show_steps` dans une interface web
- **LangGraph workflows** : chaîner plusieurs agents spécialisés
- **SqliteSaver** : persister la mémoire entre sessions